# Fine-tune the 7-site Ensemble on a new AOI

Goal: take the pre-trained **SwinIR-L1-7site** and **Prithvi-V1-L1-7site** checkpoints and adapt them to your area of interest using a small amount of additional training data.

When to use this notebook:
- You have WaPOR L3 ground truth available for some dekads of your AOI.
- The base model's out-of-the-box transferability isn't good enough for your site (e.g., your area is climatically very different from the seven African training sites, or you have a domain shift like new crops or paddy systems).
- You want to leverage what the model already learned across multiple African pilots, instead of training from scratch.

Workflow:
1. Configure the AOI + which dekads have ground truth (fine-tune split) vs which to hold out (evaluation).
2. Build input stacks (WaPOR L1 + L3 + Sentinel-2). See `inference_new_aoi.ipynb` for the data-prep pipeline if you haven't run it yet.
3. Download the pre-trained checkpoints (or use locally cached ones).
4. Fine-tune SwinIR + Prithvi with `--pretrained <ckpt>` and a **reduced learning rate** for 20-30 epochs.
5. Evaluate the fine-tuned models + ensemble on the hold-out dekads.
6. Compare against the zero-shot baseline (the unmodified 7-site model on the same hold-out).

## 1. Configure

In [ ]:
import sys, subprocess, time
from pathlib import Path

# --- paths ---
REPO = Path(r'c:\Users\z.kiala\Documents\wapor-downscale').resolve()
PY   = Path(r'C:\envs\wapor-downscale\python.exe')      # adjust to your env

# --- pre-trained checkpoints (the operational 7-site Ensemble) ---
PRETRAINED_SWINIR  = REPO / 'models' / 'multi7_swinir_l1_e96_w16' / 'swinir_best.pt'
PRETRAINED_PRITHVI = REPO / 'models' / 'multi7_prithvi_v1_l1'     / 'prithvi_best.pt'

# --- new site ---
SITE_NAME  = 'NEWSITE'                  # short tag
STACK_L1   = REPO / 'data' / SITE_NAME.lower() / 'stacks' / f'{SITE_NAME}_STACK_S2_MATCH_L3_20M_L1_FULL_1'
TRAIN_YEAR_MAX = 2023                   # dekads with year <= 2023 used for fine-tuning; year > 2023 held out

# --- fine-tune hyperparams (lower LR + fewer epochs than from-scratch) ---
FT_EPOCHS_SWINIR   = 30
FT_LR_SWINIR       = 5e-5            # ~4x lower than from-scratch (2e-4)
FT_EPOCHS_PRITHVI  = 30
FT_LR_HEAD_PRITHVI = 5e-5
FT_LR_BACKBONE_PRITHVI = 1e-6        # ~10x lower than from-scratch

# --- output ---
OUT_SWINIR  = REPO / 'models' / f'finetune_swinir_{SITE_NAME.lower()}'
OUT_PRITHVI = REPO / 'models' / f'finetune_prithvi_{SITE_NAME.lower()}'

for p in (PRETRAINED_SWINIR, PRETRAINED_PRITHVI, STACK_L1, PY):
    assert p.exists(), f'Missing: {p}'
print('OK — paths valid.')

## 2. Fine-tune SwinIR

Key flags:
- `--pretrained <ckpt>` loads the 7-site weights and **resets the epoch counter + LR schedule** (distinct from `--resume`, which fast-forwards both to continue an interrupted run).
- `--lr 5e-5` is ~4× lower than the from-scratch 2e-4 — enough to adapt the model without erasing what it already learned.
- 30 epochs is usually enough for a single site of fine-tune data; watch the validation curve and stop earlier if it plateaus.

In [ ]:
cmd = [
    PY, '-u', '-m', 'wapor_downscale.training.train_swinir',
    '--stacks-dir',     STACK_L1,
    '--out-dir',        OUT_SWINIR,
    '--train-year-max', str(TRAIN_YEAR_MAX),
    '--embed-dim', '96', '--depths', '4,4,4,4', '--window-size', '16',
    '--epochs',         str(FT_EPOCHS_SWINIR),
    '--patches-per-epoch', '1500',
    '--val-patches', '256', '--patch', '256', '--batch-size', '2',
    '--lr',             str(FT_LR_SWINIR),
    '--pretrained',     PRETRAINED_SWINIR,
]
print('[RUN]', ' '.join(str(c) for c in cmd))
subprocess.run(cmd, cwd=REPO, check=True)

## 3. Fine-tune Prithvi-V1

Prithvi has two LR groups (backbone + head). Lower both for fine-tuning; the backbone especially benefits from a very small LR to avoid catastrophic forgetting of the HLS pre-training.

In [ ]:
cmd = [
    PY, '-u', '-m', 'wapor_downscale.training.train_prithvi',
    '--stacks-dir',     STACK_L1,
    '--out-dir',        OUT_PRITHVI,
    '--train-year-max', str(TRAIN_YEAR_MAX),
    '--model-version', 'v1', '--freeze-until', '22',
    '--epochs',         str(FT_EPOCHS_PRITHVI),
    '--patches-per-epoch', '1500',
    '--val-patches', '256', '--patch', '256', '--batch-size', '2',
    '--lr-head',       str(FT_LR_HEAD_PRITHVI),
    '--lr-backbone',   str(FT_LR_BACKBONE_PRITHVI),
    '--pretrained',    PRETRAINED_PRITHVI,
]
print('[RUN]', ' '.join(str(c) for c in cmd))
subprocess.run(cmd, cwd=REPO, check=True)

## 4. Evaluate fine-tuned models on the hold-out

Compute metrics for each fine-tuned model + the ensemble on the dekads with year > `TRAIN_YEAR_MAX`.

In [ ]:
TAG = f'{SITE_NAME.lower()}_finetuned'

subprocess.run([PY, '-m', 'wapor_downscale.inference.eval_swinir',
                '--ckpt', OUT_SWINIR / 'swinir_best.pt',
                '--site-spec', f'{STACK_L1}:{TRAIN_YEAR_MAX}',
                '--out-tag', f'swin_{TAG}'], cwd=REPO, check=True)

subprocess.run([PY, '-m', 'wapor_downscale.inference.eval_prithvi',
                '--ckpt', OUT_PRITHVI / 'prithvi_best.pt',
                '--site-spec', f'{STACK_L1}:{TRAIN_YEAR_MAX}',
                '--out-tag', f'prithvi_{TAG}'], cwd=REPO, check=True)

subprocess.run([PY, '-m', 'wapor_downscale.inference.eval_ensemble',
                '--swinir-ckpt',  OUT_SWINIR / 'swinir_best.pt',
                '--prithvi-ckpt', OUT_PRITHVI / 'prithvi_best.pt',
                '--site-spec', f'{STACK_L1}:{TRAIN_YEAR_MAX}',
                '--out-tag', f'ensemble_{TAG}', '--weight', '0.5'], cwd=REPO, check=True)

print('Per-dekad CSVs + per-site aggregates saved to models/comparisons/')

## 5. Compare fine-tuned vs zero-shot baseline

Run the pre-trained checkpoints **directly** on the same hold-out (no fine-tuning) and contrast.

In [ ]:
BASELINE_TAG = f'{SITE_NAME.lower()}_zeroshot'

subprocess.run([PY, '-m', 'wapor_downscale.inference.eval_ensemble',
                '--swinir-ckpt',  PRETRAINED_SWINIR,
                '--prithvi-ckpt', PRETRAINED_PRITHVI,
                '--site-spec', f'{STACK_L1}:{TRAIN_YEAR_MAX}',
                '--out-tag', f'ensemble_{BASELINE_TAG}', '--weight', '0.5'],
               cwd=REPO, check=True)

import json
COMP = REPO / 'models' / 'comparisons'
def stats(tag):
    p = COMP / f'fair_aggregate_ensemble_{tag}.json'
    if not p.exists(): return None
    c = json.loads(p.read_text())['combined']
    return c['rmse'], c['mae'], c['r2'], c['n_pix']

zs = stats(BASELINE_TAG)
ft = stats(TAG)
if zs and ft:
    print(f'Zero-shot Ensemble: RMSE={zs[0]:.3f} mm/dekad ({zs[0]/10:.3f} mm/day)  R2={zs[2]:.3f}')
    print(f'Fine-tuned Ensemble: RMSE={ft[0]:.3f} mm/dekad ({ft[0]/10:.3f} mm/day)  R2={ft[2]:.3f}')
    improvement = (zs[0] - ft[0]) / zs[0] * 100
    print(f'\nFine-tuning improved RMSE by {improvement:.1f}%')

## 6. Use the fine-tuned models for production inference

After fine-tuning succeeds, point `inference_new_aoi.ipynb` (or `predict_ensemble.py`) at the new checkpoints instead of the original:

In [ ]:
# Example: run prediction on every dekad of the AOI using the fine-tuned ensemble
from pathlib import Path
PRED_DIR = REPO / 'models' / 'predictions' / f'{SITE_NAME}_finetuned'
PRED_DIR.mkdir(parents=True, exist_ok=True)
for fp in sorted(STACK_L1.glob('*.tif')):
    subprocess.run([PY, '-m', 'wapor_downscale.inference.predict_ensemble',
                    '--swinir-ckpt',  OUT_SWINIR / 'swinir_best.pt',
                    '--prithvi-ckpt', OUT_PRITHVI / 'prithvi_best.pt',
                    '--stack',        fp,
                    '--out-dir',      PRED_DIR,
                    '--weight', '0.5'], cwd=REPO, check=True)

## Tips and pitfalls

### How many training dekads do I need?

- **< 20 dekads**: fine-tuning may overfit. Stick to zero-shot inference, or use the smallest LR (5e-6 backbone, 5e-5 head) and 10-15 epochs only.
- **20-50 dekads**: sweet spot. 20-30 epochs at the recommended LRs typically gives the biggest gains.
- **> 50 dekads**: you might as well also try training from scratch with `train_swinir.py` / `train_prithvi.py` without `--pretrained` — let the data tell you which works better.

### Catastrophic forgetting

If fine-tuning makes the model worse than the zero-shot baseline on the hold-out, you've over-adapted. Drop both LRs by 5× and try again.

### When NOT to fine-tune

- Your AOI is climatically inside the model's training distribution (sub-Saharan + Mediterranean Africa) AND you don't have ground truth — skip fine-tuning, just use the pre-trained model.
- Your hold-out RMSE is already < 0.45 mm/day with the zero-shot model — the gain from fine-tuning is usually <10% in this regime.

### When to retrain from scratch instead

- Your AOI is extra-continental (Asia, Latin America) and you have hundreds of dekads. The model's African priors may actively hurt — train fresh.
- You want to add a new architectural variant (e.g., a different encoder) — the pretrained head's shape won't match.

### Saving + sharing

The fine-tuned checkpoint is fully self-contained (no need to ship the original 7-site one along with it). You can upload your `finetune_<site>/swinir_best.pt` and `finetune_<site>/prithvi_best.pt` to Hugging Face / Zenodo and others can use them directly via `inference_new_aoi.ipynb` by pointing the model paths at the URL or local cache.